In [1]:
import os
import joblib
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from xgboost import XGBClassifier

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(__file__).resolve().parent.parent

DATA_DIR = BASE_DIR / "data"

df = pd.read_csv(DATA_DIR / "cleaned_data.csv")

(300491, 23)


,Accident_Index,Accident Date,Month,Day_of_Week,Year,Junction_Control,Junction_Detail,Accident_Severity,Latitude,Light_Conditions,...,Number_of_Casualties,Number_of_Vehicles,Police_Force,Road_Surface_Conditions,Road_Type,Speed_limit,Time,Urban_or_Rural_Area,Weather_Conditions,Vehicle_Type
0,200901BS70315,2021-06-16,Jun,Tuesday,2021,Give way or uncontrolled,T or staggered junction,Slight,51.486145,Daylight,...,1,1,Metropolitan Police,Dry,Single carriageway,30,2026-07-30T12:34:00.000+05:30,Urban,Fine no high winds,Car
1,200901BS70436,2021-07-08,Aug,Friday,2021,Give way or uncontrolled,T or staggered junction,Slight,51.492829,Daylight,...,1,1,Metropolitan Police,Dry,Single carriageway,30,2026-07-30T15:15:00.000+05:30,Urban,Fine no high winds,Car
2,200901CP00285,2021-10-30,Oct,Friday,2021,Give way or uncontrolled,T or staggered junction,Slight,51.510703,Daylight,...,1,1,City of London,Dry,Single carriageway,30,2026-07-30T08:50:00.000+05:30,Urban,Fine no high winds,Car
3,200901CW10181,2021-01-18,Jan,Sunday,2021,Auto traffic signal,Crossroads,Slight,51.492670,Daylight,...,1,2,Metropolitan Police,Wet or damp,Single carriageway,30,2026-07-30T13:05:00.000+05:30,Urban,Fine no high winds,Car
4,200901CW10194,2021-11-02,Feb,Wednesday,2021,Give way or uncontrolled,T or staggered junction,Slight,51.485691,Daylight,...,1,2,Metropolitan Police,Dry,Single carriageway,30,2026-07-30T06:36:00.000+05:30,Urban,Fine no high winds,Car


In [3]:
# Check class distribution
print(df["Accident_Severity"].value_counts())

# Split by class
slight = df[df["Accident_Severity"] == "Slight"]
serious = df[df["Accident_Severity"] == "Serious"]
fatal = df[df["Accident_Severity"] == "Fatal"]

# Partial oversampling
serious = serious.sample(
    n=100000,
    replace=True,
    random_state=42
)

fatal = fatal.sample(
    n=50000,
    replace=True,
    random_state=42
)

# Combine
df = pd.concat(
    [slight, serious, fatal],
    ignore_index=True
)

# Shuffle
df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nBalanced Dataset")
print(df["Accident_Severity"].value_counts())

Accident_Severity
Slight     256516
Serious     40083
Fatal        3892
Name: count, dtype: int64

Balanced Dataset
Accident_Severity
Slight     256516
Serious    100000
Fatal       50000
Name: count, dtype: int64


In [4]:
drop_columns = [
    "Accident_Index",
    "Accident Date","Latitude",
    "Longitude"

]

df = df.drop(columns=drop_columns)

df.head()

,Month,Day_of_Week,Year,Junction_Control,Junction_Detail,Accident_Severity,Light_Conditions,Local_Authority_(District),Carriageway_Hazards,Number_of_Casualties,Number_of_Vehicles,Police_Force,Road_Surface_Conditions,Road_Type,Speed_limit,Time,Urban_or_Rural_Area,Weather_Conditions,Vehicle_Type
0,Feb,Wednesday,2021,Give way or uncontrolled,Roundabout,Serious,Daylight,Liverpool,NaN,2,2,Merseyside,Wet or damp,Single carriageway,30,2026-07-30T16:25:00.000+05:30,Urban,Other,Car
1,May,Sunday,2021,Give way or uncontrolled,T or staggered junction,Slight,Daylight,Manchester,NaN,1,2,Greater Manchester,Dry,Single carriageway,30,2026-07-30T21:25:00.000+05:30,Urban,Fine no high winds,Car
2,Dec,Tuesday,2021,Auto traffic signal,T or staggered junction,Slight,Darkness - lights lit,Wigan,NaN,2,2,Greater Manchester,Dry,Single carriageway,30,2026-07-30T17:00:00.000+05:30,Urban,Fine no high winds,Car
3,May,Friday,2022,Give way or uncontrolled,T or staggered junction,Slight,Daylight,Waltham Forest,NaN,1,1,Metropolitan Police,Dry,Single carriageway,30,2026-07-30T11:15:00.000+05:30,Urban,Fine no high winds,Motorcycle 125cc and under
4,Oct,Monday,2022,Data missing or out of range,Not at junction or within 20 metres,Slight,Darkness - lights lit,North Warwickshire,NaN,1,3,Warwickshire,Wet or damp,Dual carriageway,70,2026-07-30T17:25:00.000+05:30,Rural,Fine + high winds,Car


In [5]:
df["Hour"] = pd.to_datetime(
    df["Time"],
    errors="coerce"
).dt.hour

print(df["Hour"].isna().sum())

df = df.drop(columns=["Time"])

0


In [6]:
label_encoder = LabelEncoder()

df["Accident_Severity"] = label_encoder.fit_transform(
    df["Accident_Severity"]
)

print(label_encoder.classes_)

df.head()

['Fatal' 'Serious' 'Slight']


,Month,Day_of_Week,Year,Junction_Control,Junction_Detail,Accident_Severity,Light_Conditions,Local_Authority_(District),Carriageway_Hazards,Number_of_Casualties,Number_of_Vehicles,Police_Force,Road_Surface_Conditions,Road_Type,Speed_limit,Urban_or_Rural_Area,Weather_Conditions,Vehicle_Type,Hour
0,Feb,Wednesday,2021,Give way or uncontrolled,Roundabout,1,Daylight,Liverpool,NaN,2,2,Merseyside,Wet or damp,Single carriageway,30,Urban,Other,Car,16
1,May,Sunday,2021,Give way or uncontrolled,T or staggered junction,2,Daylight,Manchester,NaN,1,2,Greater Manchester,Dry,Single carriageway,30,Urban,Fine no high winds,Car,21
2,Dec,Tuesday,2021,Auto traffic signal,T or staggered junction,2,Darkness - lights lit,Wigan,NaN,2,2,Greater Manchester,Dry,Single carriageway,30,Urban,Fine no high winds,Car,17
3,May,Friday,2022,Give way or uncontrolled,T or staggered junction,2,Daylight,Waltham Forest,NaN,1,1,Metropolitan Police,Dry,Single carriageway,30,Urban,Fine no high winds,Motorcycle 125cc and under,11
4,Oct,Monday,2022,Data missing or out of range,Not at junction or within 20 metres,2,Darkness - lights lit,North Warwickshire,NaN,1,3,Warwickshire,Wet or damp,Dual carriageway,70,Rural,Fine + high winds,Car,17


In [7]:
X = df.drop(columns=["Accident_Severity"])

y = df["Accident_Severity"]

print(X.shape)
print(y.shape)

(406516, 18)
(406516,)


In [8]:
categorical_columns = X.select_dtypes(
    include="object"
).columns.tolist()

numerical_columns = X.select_dtypes(
    exclude="object"
).columns.tolist()

print(categorical_columns)

print()

print(numerical_columns)

['Month', 'Day_of_Week', 'Junction_Control', 'Junction_Detail', 'Light_Conditions', 'Local_Authority_(District)', 'Carriageway_Hazards', 'Police_Force', 'Road_Surface_Conditions', 'Road_Type', 'Urban_or_Rural_Area', 'Weather_Conditions', 'Vehicle_Type']

['Year', 'Number_of_Casualties', 'Number_of_Vehicles', 'Speed_limit', 'Hour']


In [9]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Numerical preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_columns),
        ("cat", categorical_transformer, categorical_columns)
    ]
)

preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training :", X_train.shape)
print("Testing  :", X_test.shape)

Training : (325212, 18)
Testing  : (81304, 18)


In [11]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

(325212, 557)
(81304, 557)


In [12]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl"
)

print("Preprocessor Saved Successfully")

Preprocessor Saved Successfully


In [13]:
print(X_train_processed.shape)
print(X_test_processed.shape)

(325212, 557)
(81304, 557)


In [14]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_processed, y_train)

print("Random Forest Training Completed")

Random Forest Training Completed


In [15]:
rf_predictions = rf_model.predict(X_test_processed)

rf_accuracy = accuracy_score(y_test, rf_predictions)

rf_precision = precision_score(
    y_test,
    rf_predictions,
    average="weighted"
)

rf_recall = recall_score(
    y_test,
    rf_predictions,
    average="weighted"
)

rf_f1 = f1_score(
    y_test,
    rf_predictions,
    average="weighted"
)

print(f"Accuracy  : {rf_accuracy:.4f}")
print(f"Precision : {rf_precision:.4f}")
print(f"Recall    : {rf_recall:.4f}")
print(f"F1 Score  : {rf_f1:.4f}")

Accuracy  : 0.6312
Precision : 0.5212
Recall    : 0.6312
F1 Score  : 0.4887


/home/dockeradmin/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [16]:
joblib.dump(
    rf_model,
    "../models/random_forest.pkl",
    compress=3
)

print("Random Forest Saved")

Random Forest Saved


In [17]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

# Compute balanced sample weights
sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

# XGBoost Model
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),

    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    min_child_weight=3,
    gamma=0.2,

    random_state=42,
    eval_metric="mlogloss",

    n_jobs=-1
)

# Train
xgb_model.fit(
    X_train_processed,
    y_train,
    sample_weight=sample_weights
)

print("XGBoost Training Completed")

XGBoost Training Completed


In [18]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

pred = xgb_model.predict(X_test_processed)

print("Accuracy :", accuracy_score(y_test, pred))
print(
    "Precision:",
    precision_score(y_test, pred, average="weighted")
)
print(
    "Recall:",
    recall_score(y_test, pred, average="weighted")
)
print(
    "F1 Score:",
    f1_score(y_test, pred, average="weighted")
)

print("\nClassification Report\n")
print(
    classification_report(
        y_test,
        pred,
        target_names=label_encoder.classes_
    )
)

Accuracy : 0.6024426842467775
Precision: 0.6536357153856976
Recall: 0.6024426842467775
F1 Score: 0.614449256260354

Classification Report

              precision    recall  f1-score   support

       Fatal       0.42      0.79      0.55     10000
     Serious       0.40      0.43      0.41     20000
      Slight       0.80      0.63      0.71     51304

    accuracy                           0.60     81304
   macro avg       0.54      0.62      0.56     81304
weighted avg       0.65      0.60      0.61     81304



In [19]:
joblib.dump(
    xgb_model,
    "../models/xgboost_model.pkl",
    compress=3
)

print("XGBoost Model Saved")

XGBoost Model Saved


In [20]:
joblib.dump(
    label_encoder,
    "../models/label_encoder.pkl",
    compress=3
)

print("Label Encoder Saved")

Label Encoder Saved


In [21]:
feature_names = preprocessor.get_feature_names_out()

joblib.dump(
    feature_names,
    "../models/feature_columns.pkl",
    compress=3
)

print("Feature Columns Saved")

Feature Columns Saved


In [22]:
joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl",
    compress=3
)

print("Preprocessor Saved")

Preprocessor Saved


In [24]:
from sklearn.metrics import classification_report

pred = rf_model.predict(X_test_processed)

print(classification_report(
    y_test,
    pred,
    target_names=label_encoder.classes_
))

              precision    recall  f1-score   support

       Fatal       1.00      0.00      0.00     10000
     Serious       0.00      0.00      0.00     20000
      Slight       0.63      1.00      0.77     51304

    accuracy                           0.63     81304
   macro avg       0.54      0.33      0.26     81304
weighted avg       0.52      0.63      0.49     81304



/home/dockeradmin/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/dockeradmin/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/dockeradmin/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

In [25]:
from sklearn.metrics import classification_report

pred = xgb_model.predict(X_test_processed)

print(classification_report(
    y_test,
    pred,
    target_names=label_encoder.classes_
))

              precision    recall  f1-score   support

       Fatal       0.42      0.79      0.55     10000
     Serious       0.40      0.43      0.41     20000
      Slight       0.80      0.63      0.71     51304

    accuracy                           0.60     81304
   macro avg       0.54      0.62      0.56     81304
weighted avg       0.65      0.60      0.61     81304



In [26]:
import pandas as pd

# Predict on test data
pred = xgb_model.predict(X_test_processed)

# Convert predictions back to labels
pred_labels = label_encoder.inverse_transform(pred)

# Find rows predicted as Serious
serious_rows = X_test[pred_labels == "Serious"].copy()

print("Number of Serious predictions:", len(serious_rows))

serious_rows.head(10)

Number of Serious predictions: 21925


,Month,Day_of_Week,Year,Junction_Control,Junction_Detail,Light_Conditions,Local_Authority_(District),Carriageway_Hazards,Number_of_Casualties,Number_of_Vehicles,Police_Force,Road_Surface_Conditions,Road_Type,Speed_limit,Urban_or_Rural_Area,Weather_Conditions,Vehicle_Type,Hour
360600,Jul,Wednesday,2021,Data missing or out of range,Not at junction or within 20 metres,Daylight,Fife,NaN,1,1,Fife,Dry,Single carriageway,20,Rural,Fine no high winds,Car,14
105463,May,Saturday,2022,Data missing or out of range,Not at junction or within 20 metres,Daylight,Lincoln,NaN,1,1,Lincolnshire,Dry,Single carriageway,30,Urban,Fine no high winds,Van / Goods 3.5 tonnes mgw or under,19
350274,Jan,Thursday,2022,Give way or uncontrolled,Mini-roundabout,Darkness - lights lit,Dudley,NaN,1,1,West Midlands,Wet or damp,Single carriageway,30,Urban,Fine no high winds,Car,17
43447,Feb,Sunday,2021,Data missing or out of range,Not at junction or within 20 metres,Darkness - lights lit,Havant,NaN,1,2,Hampshire,Wet or damp,Single carriageway,30,Urban,Fine no high winds,Car,18
307966,May,Wednesday,2021,Give way or uncontrolled,T or staggered junction,Daylight,Bromley,NaN,1,1,Metropolitan Police,Dry,Single carriageway,30,Urban,Fine no high winds,Car,9
302456,Dec,Thursday,2021,Give way or uncontrolled,T or staggered junction,Daylight,Barnet,NaN,1,1,Metropolitan Police,Dry,Single carriageway,30,Urban,Fine no high winds,Car,12
126087,Oct,Saturday,2021,Give way or uncontrolled,Mini-roundabout,Daylight,South Lanarkshire,NaN,1,1,Strathclyde,Dry,Single carriageway,30,Urban,Fine no high winds,Van / Goods 3.5 tonnes mgw or under,16
146504,Mar,Tuesday,2022,Data missing or out of range,Not at junction or within 20 metres,Daylight,North Somerset,NaN,1,1,Avon and Somerset,Dry,Single carriageway,30,Urban,Fine no high winds,Car,15
7712,Jul,Saturday,2021,Not at junction or within 20 metres,Not at junction or within 20 metres,Daylight,Wealden,NaN,1,2,Sussex,Dry,Single carriageway,30,Rural,Fine no high winds,Motorcycle over 500cc,19
135633,Jul,Monday,2022,Not at junction or within 20 metres,Not at junction or within 20 metres,Daylight,Chelmsford,NaN,1,2,Essex,Dry,Single carriageway,30,Urban,Fine no high winds,Car,18


In [27]:
result = serious_rows.copy()
result["Actual_Severity"] = label_encoder.inverse_transform(
    y_test.loc[serious_rows.index]
)

result.head(10)

,Month,Day_of_Week,Year,Junction_Control,Junction_Detail,Light_Conditions,Local_Authority_(District),Carriageway_Hazards,Number_of_Casualties,Number_of_Vehicles,Police_Force,Road_Surface_Conditions,Road_Type,Speed_limit,Urban_or_Rural_Area,Weather_Conditions,Vehicle_Type,Hour,Actual_Severity
360600,Jul,Wednesday,2021,Data missing or out of range,Not at junction or within 20 metres,Daylight,Fife,NaN,1,1,Fife,Dry,Single carriageway,20,Rural,Fine no high winds,Car,14,Slight
105463,May,Saturday,2022,Data missing or out of range,Not at junction or within 20 metres,Daylight,Lincoln,NaN,1,1,Lincolnshire,Dry,Single carriageway,30,Urban,Fine no high winds,Van / Goods 3.5 tonnes mgw or under,19,Slight
350274,Jan,Thursday,2022,Give way or uncontrolled,Mini-roundabout,Darkness - lights lit,Dudley,NaN,1,1,West Midlands,Wet or damp,Single carriageway,30,Urban,Fine no high winds,Car,17,Slight
43447,Feb,Sunday,2021,Data missing or out of range,Not at junction or within 20 metres,Darkness - lights lit,Havant,NaN,1,2,Hampshire,Wet or damp,Single carriageway,30,Urban,Fine no high winds,Car,18,Slight
307966,May,Wednesday,2021,Give way or uncontrolled,T or staggered junction,Daylight,Bromley,NaN,1,1,Metropolitan Police,Dry,Single carriageway,30,Urban,Fine no high winds,Car,9,Serious
302456,Dec,Thursday,2021,Give way or uncontrolled,T or staggered junction,Daylight,Barnet,NaN,1,1,Metropolitan Police,Dry,Single carriageway,30,Urban,Fine no high winds,Car,12,Serious
126087,Oct,Saturday,2021,Give way or uncontrolled,Mini-roundabout,Daylight,South Lanarkshire,NaN,1,1,Strathclyde,Dry,Single carriageway,30,Urban,Fine no high winds,Van / Goods 3.5 tonnes mgw or under,16,Serious
146504,Mar,Tuesday,2022,Data missing or out of range,Not at junction or within 20 metres,Daylight,North Somerset,NaN,1,1,Avon and Somerset,Dry,Single carriageway,30,Urban,Fine no high winds,Car,15,Slight
7712,Jul,Saturday,2021,Not at junction or within 20 metres,Not at junction or within 20 metres,Daylight,Wealden,NaN,1,2,Sussex,Dry,Single carriageway,30,Rural,Fine no high winds,Motorcycle over 500cc,19,Serious
135633,Jul,Monday,2022,Not at junction or within 20 metres,Not at junction or within 20 metres,Daylight,Chelmsford,NaN,1,2,Essex,Dry,Single carriageway,30,Urban,Fine no high winds,Car,18,Slight
